In [1]:
### Count number of repos with incompatible licenses

import os
import glob

# Directory containing CSV files
RESULTS_FOLDER = "../shell_scripts/results"

# Get all CSV files in the results folder
csv_files = glob.glob(os.path.join(RESULTS_FOLDER, "*.csv"))

total_file = 0
total_conflict = 0

for file in csv_files:
    
    num_conflict = int(file.split('_')[-1].split('.')[0])
    if num_conflict >= 1:
        print(f"Reading {file}")
        total_file = total_file+1
        total_conflict += num_conflict

print(f"Number of unique repos with a conflict: {total_file}\nTotal number of conflicting methods: {total_conflict}")

Reading ../shell_scripts/results/microsoft_MixedRealityToolkit-Unity_matches_514203695_49.csv
Reading ../shell_scripts/results/microsoft_elasticsearch_matches_302933980_2.csv
Reading ../shell_scripts/results/microsoft_SCVMMLinuxGuestAgent_matches_1017163135_1.csv
Reading ../shell_scripts/results/microsoft_WinObjC_matches_2233234028_17.csv
Reading ../shell_scripts/results/microsoft_nodejstools_matches_1769725023_1.csv
Reading ../shell_scripts/results/microsoft_TypeScript-Sublime-Plugin_matches_1609725217_1.csv
Reading ../shell_scripts/results/microsoft_vscode_matches_2322838020_36.csv
Reading ../shell_scripts/results/microsoft_DirectXTex_matches_1451294112_5.csv
Reading ../shell_scripts/results/microsoft_lis-tempest-old_matches_1569102924_1.csv
Reading ../shell_scripts/results/microsoft_LIS3.5_matches_415658888_87.csv
Reading ../shell_scripts/results/microsoft_spark_matches_696138832_9.csv
Reading ../shell_scripts/results/microsoft_deep-space_matches_1271148833_7.csv
Reading ../shell_sc

In [ ]:
'''
import os
import glob
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

# Database connection details
DB_NAME = "github_repos"
DB_USER = "postgres"
DB_PASSWORD = "Sphings@19"
DB_HOST = "localhost"
DB_PORT = "5432"

# Directory containing CSV files
RESULTS_FOLDER = "../results"

try:
    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cur = conn.cursor()

    # Get all CSV files in the results folder
    csv_files = glob.glob(os.path.join(RESULTS_FOLDER, "*.csv"))

    # Read and combine all CSV files into a single DataFrame
    all_data = pd.DataFrame()

    for file in csv_files:
        print(f"Reading {file}...")
        df = pd.read_csv(file)

        # Rename CSV columns to match the database
        df.rename(columns={
            'Hash': 'hash',
            'Project ID': 'project_id',
            'Version': 'version',
            'License': 'license',
            'Method Name': 'method_name',
            'File Location': 'file_location',
            'Function Code': 'function_code',
            'Repository URL': 'repository_url',
            'Query Project': 'query_project',
            'Violation': 'violation',
            'Source_project': 'Source_project',
            'Source_project_version':'Source_project_version'
        }, inplace=True)

        all_data = pd.concat([all_data, df], ignore_index=True)

    # Remove duplicates based on (hash, project_id)
    all_data.drop_duplicates(subset=['hash', 'project_id'], inplace=True)

    # Generate unique ID by combining hash and project_id
    all_data['_id'] = all_data['hash'].astype(str) + "_" + all_data['project_id'].astype(str)

    # Convert DataFrame to a list of tuples for batch insert
    records_to_insert = [
        (
            row['_id'], row['hash'], row['project_id'], row['version'], row['license'], row['method_name'],
            row['file_location'], row['function_code'], row['repository_url'], row['query_project'], row['violation'],
            row['Source_project'],row['Source_project_version']
        ) for _, row in all_data.iterrows()
    ]

    # Insert all records in bulk
    insert_query = """
     INSERT INTO repository_data (
        _id, hash, project_id, version, license, method_name,
        file_location, function_code, repository_url, query_project, violation, Source_project, Source_project_version
    ) VALUES %s
    ON CONFLICT (hash, project_id, version) DO NOTHING;
    """
    
    
    execute_values(cur, insert_query, records_to_insert)

    # Commit changes
    conn.commit()
    print(f"Inserted {len(records_to_insert)} new records successfully.")

except Exception as e:
    print("Error:", e)

finally:
    # Close connection
    if conn:
        cur.close()
        conn.close()
'''